<a href="https://colab.research.google.com/github/afcallenderTTU/AC-TTU-web/blob/master/radian_mzml_analysis_current.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Radian ASAP mzML analysis notebook

This notebook analyzes Waters Radian ASAP data that have already been converted from MassLynx RAW to **mzML** with ProteoWizard.

It is designed to:

- read all `.mzML` files from a Google Drive folder
- separate scans by **Waters function number**
- extract **TIC** and **XIC** traces for chosen ions
- calculate, within the first **60 seconds**:
  - **peak signal**
  - **integrated signal**
- build **background-subtracted summed spectra**
- centroid the **summed/background-subtracted spectra** for peak inspection
- compare **integrated ion ratios** with **trimmed scan-by-scan ratios**
- write CSV summaries back to Google Drive

In [ ]:
!pip -q install pyteomics lxml pandas matplotlib numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 5.7 MB/s eta 0:00:00


## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Settings and file discovery

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyteomics import mzml
from scipy.signal import find_peaks

DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Radian')

FUNCTIONS_TO_USE = [1, 2]
MZ_TOL = 0.5
ANALYSIS_END_S = 60.0
BACKGROUND_END_S = 10.0
MZ_RANGE = (50.0, 350.0)
BIN_WIDTH = 0.05

TARGETS = {
    "benzaldehyde_106": 106.0,
    "bpa_diacetate_MH_313": 313.0,
    "bpa_fragment_213": 213.0,
    "bpa_diacetate_fragment_270": 270.0,
}

mzml_files = sorted(DATA_DIR.glob("*.mzML"))
print(f"Found {len(mzml_files)} mzML files")
for f in mzml_files[:10]:
    print(" -", f.name)

Found 30 mzML files
 - (1) 100 BPA - 500 BZA.mzML
 - (1) 100 BPA - 900 BZA.mzML
 - (1) 100 BPA - 900 Tol.mzML
 - (2) 100 BPA - 500 BZA.mzML
 - (2) 100 BPA - 900 BZA.mzML
 - (2) 100 BPA - 900 Tol.mzML
 - (3) 100 BPA - 500 BZA.mzML
 - (3) 100 BPA - 900 BZA.mzML
 - (3) 100 BPA - 900 Tol.mzML
 - 100 BPA - 100 BZA - 800 Tol.mzML


## Helper functions

In [ ]:
function_re = re.compile(r'function=(\d+)')

def get_function_number(spec):
    sid = spec.get("id", "")
    m = function_re.search(sid)
    if m:
        return int(m.group(1))

    scan_list = spec.get("scanList", {}).get("scan", [])
    if scan_list:
        scan = scan_list[0]
        if "preset scan configuration" in scan:
            try:
                return int(scan["preset scan configuration"])
            except Exception:
                pass
    return None

def get_rt_seconds(spec):
    scan_list = spec.get("scanList", {}).get("scan", [])
    if not scan_list:
        return np.nan
    scan = scan_list[0]
    rt = scan.get("scan start time")
    if rt is None:
        return np.nan
    return float(rt) * 60.0

def xic_intensity(mz_array, inten_array, center, tol=0.5):
    mask = (mz_array >= center - tol) & (mz_array <= center + tol)
    if not np.any(mask):
        return 0.0
    return float(np.sum(inten_array[mask]))

def integrate_trace(rt_s, intensity):
    if len(rt_s) < 2:
        return 0.0
    order = np.argsort(rt_s)
    x = np.asarray(rt_s)[order]
    y = np.asarray(intensity)[order]
    return float(np.trapz(y, x))

def sum_spectra(spectra, mz_min=MZ_RANGE[0], mz_max=MZ_RANGE[1], bin_width=BIN_WIDTH):
    edges = np.arange(mz_min, mz_max + bin_width, bin_width)
    summed = np.zeros(len(edges) - 1, dtype=float)

    for s in spectra:
        mz = s["mz"]
        intensity = s["intensity"]
        mask = (mz >= mz_min) & (mz <= mz_max)
        hist, _ = np.histogram(mz[mask], bins=edges, weights=intensity[mask])
        summed += hist

    centers = (edges[:-1] + edges[1:]) / 2
    return centers, summed

def centroid_from_profile(mz_axis, intensity, prominence=None, height=None, distance=2):
    y = np.asarray(intensity, dtype=float)

    if len(y) == 0 or np.all(y <= 0):
        return np.array([]), np.array([])

    if prominence is None:
        prominence = max(np.max(y) * 0.01, 1.0)

    peaks, props = find_peaks(y, prominence=prominence, height=height, distance=distance)

    centroid_mz = []
    centroid_intensity = []

    for p in peaks:
        left = p
        right = p

        while left > 0 and y[left] > 0.1 * y[p]:
            left -= 1
        while right < len(y) - 1 and y[right] > 0.1 * y[p]:
            right += 1

        mz_slice = mz_axis[left:right+1]
        y_slice = y[left:right+1]

        area = np.sum(y_slice)
        if area <= 0:
            continue

        mz_centroid = np.sum(mz_slice * y_slice) / area
        centroid_mz.append(float(mz_centroid))
        centroid_intensity.append(float(np.max(y_slice)))

    return np.array(centroid_mz), np.array(centroid_intensity)

## Load all mzML files

In [ ]:
records = []
raw_spectra = []

for mzml_path in mzml_files:
    sample_name = mzml_path.stem
    print("Reading:", sample_name)

    with mzml.MzML(str(mzml_path)) as reader:
        for spec in reader:
            if int(spec.get("ms level", 0)) != 1:
                continue

            fn = get_function_number(spec)
            if fn not in FUNCTIONS_TO_USE:
                continue

            rt_s = get_rt_seconds(spec)
            if np.isnan(rt_s) or rt_s > ANALYSIS_END_S:
                continue

            mz_array = np.asarray(spec.get("m/z array", []), dtype=float)
            inten_array = np.asarray(spec.get("intensity array", []), dtype=float)

            if mz_array.size == 0 or inten_array.size == 0:
                continue

            tic = float(spec.get("total ion current", np.sum(inten_array)))

            row = {
                "sample": sample_name,
                "function": fn,
                "rt_s": rt_s,
                "tic": tic,
            }

            for label, center in TARGETS.items():
                row[label] = xic_intensity(mz_array, inten_array, center, MZ_TOL)

            records.append(row)
            raw_spectra.append({
                "sample": sample_name,
                "function": fn,
                "rt_s": rt_s,
                "mz": mz_array,
                "intensity": inten_array,
            })

traces_df = pd.DataFrame(records).sort_values(["sample", "function", "rt_s"]).reset_index(drop=True)

print()
print("Total spectra loaded:", len(raw_spectra))
print("Samples found:", traces_df["sample"].nunique() if len(traces_df) else 0)
traces_df.head()

Reading: (1) 100 BPA - 500 BZA
Reading: (1) 100 BPA - 900 BZA
Reading: (1) 100 BPA - 900 Tol
Reading: (2) 100 BPA - 500 BZA
Reading: (2) 100 BPA - 900 BZA
Reading: (2) 100 BPA - 900 Tol
Reading: (3) 100 BPA - 500 BZA
Reading: (3) 100 BPA - 900 BZA
Reading: (3) 100 BPA - 900 Tol
Reading: 100 BPA - 100 BZA - 800 Tol
Reading: 100 BPA - 250 BZA - 650 Tol
Reading: 100 BPA - 500 BZA - 400 Tol
Reading: 100 BPA - 750 BZA - 150 Tol
Reading: 100 BPA - 900 BZA
Reading: 100 BZA - 900 Tol
Reading: 500 BPA - 100 BZA - 400 Tol
Reading: 500 BPA - 250 BZA - 250 Tol
Reading: 500 BPA - 500 BZA
Reading: APT (1) 100 BPA - 900 BZA
Reading: APT (1) 100 BPA - 900 Tol
Reading: APT
Reading: BCA Candle 
Reading: BCW_toluene
Reading: BCW_toluene_temp
Reading: Bispehnol Standard Test 
Reading: Bl (1) 100 BPA - 900 BZA
Reading: Bl (1) 100 BPA - 900 Tol
Reading: Bl (2) 100 BPA - 900 Tol
Reading: Blank Candle
Reading: White_toluene

Total spectra loaded: 18258
Samples found: 30


,sample,function,rt_s,tic,benzaldehyde_106,bpa_diacetate_MH_313,bpa_fragment_213,bpa_diacetate_fragment_270
0,(1) 100 BPA - 500 BZA,1,0.047,25990676.0,0.0,0.0,0.0,237092.0
1,(1) 100 BPA - 500 BZA,1,0.247,25207988.0,0.0,180976.0,0.0,127796.0
2,(1) 100 BPA - 500 BZA,1,0.447,28085444.0,0.0,142132.0,0.0,145204.0
3,(1) 100 BPA - 500 BZA,1,0.647,26054224.0,0.0,109364.0,0.0,313960.0
4,(1) 100 BPA - 500 BZA,1,0.847,25290460.0,0.0,0.0,0.0,95096.0


## First-minute quantitation summary

In [ ]:
summary_rows = []

for (sample, fn), sub in traces_df.groupby(["sample", "function"]):
    sub = sub.sort_values("rt_s")

    for label in TARGETS:
        peak_signal = float(sub[label].max()) if len(sub) else 0.0
        peak_time_s = float(sub.loc[sub[label].idxmax(), "rt_s"]) if len(sub) else np.nan
        integrated_signal = integrate_trace(sub["rt_s"].values, sub[label].values)

        summary_rows.append({
            "sample": sample,
            "function": fn,
            "target": label,
            "target_mz": TARGETS[label],
            "peak_signal_0_60s": peak_signal,
            "peak_time_s": peak_time_s,
            "integrated_signal_0_60s": integrated_signal,
        })

summary_df = pd.DataFrame(summary_rows).sort_values(["sample", "function", "target"]).reset_index(drop=True)
summary_df.head(20)

/tmp/ipykernel_467/2267992279.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(y, x))
/tmp/ipykernel_467/2267992279.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(y, x))
/tmp/ipykernel_467/2267992279.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(y, x))
/tmp/ipykernel_467/2267992279.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(y, x))
/tmp/ipykernel_467/2267992279.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return fl

,sample,function,target,target_mz,peak_signal_0_60s,peak_time_s,integrated_signal_0_60s
0,(1) 100 BPA - 500 BZA,1,benzaldehyde_106,106.0,35814528.0,10.243000,1.373417e+08
1,(1) 100 BPA - 500 BZA,1,bpa_diacetate_MH_313,313.0,178575616.0,31.235000,4.864583e+09
2,(1) 100 BPA - 500 BZA,1,bpa_diacetate_fragment_270,270.0,24405352.0,10.643000,8.548089e+07
3,(1) 100 BPA - 500 BZA,1,bpa_fragment_213,213.0,29292880.0,13.442000,1.421118e+08
4,(1) 100 BPA - 500 BZA,2,benzaldehyde_106,106.0,38913136.0,13.692000,2.497614e+08
5,(1) 100 BPA - 500 BZA,2,bpa_diacetate_MH_313,313.0,140692224.0,44.679000,4.047890e+09
6,(1) 100 BPA - 500 BZA,2,bpa_diacetate_fragment_270,270.0,4616224.0,15.291000,8.767890e+07
7,(1) 100 BPA - 500 BZA,2,bpa_fragment_213,213.0,39869168.0,10.493000,2.056833e+08
8,(1) 100 BPA - 900 BZA,1,benzaldehyde_106,106.0,14930160.0,10.642000,9.410877e+07
9,(1) 100 BPA - 900 BZA,1,bpa_diacetate_MH_313,313.0,191633728.0,29.035000,5.606425e+09


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=summary_df)

https://docs.google.com/spreadsheets/d/1gtXm-jD1h2vDvWeXIwv-TBoSsbRFoxtFzVSAp5abGHA/edit#gid=0


## Summed spectra, background subtraction, and centroiding

In [ ]:
bgsub_spectra = {}
peak_check_rows = []

for (sample, fn), _ in traces_df.groupby(["sample", "function"]):
    fn_specs = [s for s in raw_spectra if s["sample"] == sample and s["function"] == fn]
    bg_specs = [s for s in fn_specs if s["rt_s"] <= BACKGROUND_END_S]
    sig_specs = [s for s in fn_specs if BACKGROUND_END_S < s["rt_s"] <= ANALYSIS_END_S]

    x_bg, y_bg = sum_spectra(bg_specs)
    x_sig, y_sig = sum_spectra(sig_specs)

    scale = len(sig_specs) / len(bg_specs) if len(bg_specs) > 0 else 0.0
    y_bg_scaled = y_bg * scale
    y_sub = y_sig - y_bg_scaled

    c_mz, c_int = centroid_from_profile(x_sig, y_sub)

    bgsub_spectra[(sample, fn)] = {
        "mz": x_sig,
        "signal_sum": y_sig,
        "background_scaled": y_bg_scaled,
        "background_subtracted": y_sub,
        "centroid_mz": c_mz,
        "centroid_intensity": c_int,
    }

    for label, center in TARGETS.items():
        mask = (c_mz >= center - MZ_TOL) & (c_mz <= center + MZ_TOL)
        if np.any(mask):
            idx = np.where(mask)[0]
            best = idx[np.argmax(c_int[idx])]
            obs_mz = float(c_mz[best])
            obs_int = float(c_int[best])
        else:
            obs_mz = np.nan
            obs_int = 0.0

        peak_check_rows.append({
            "sample": sample,
            "function": fn,
            "target": label,
            "nominal_mz": center,
            "observed_centroid_mz": obs_mz,
            "centroid_peak_intensity": obs_int,
        })

peak_check_df = pd.DataFrame(peak_check_rows).sort_values(["sample", "function", "target"]).reset_index(drop=True)
peak_check_df.head(20)

,sample,function,target,nominal_mz,observed_centroid_mz,centroid_peak_intensity
0,(1) 100 BPA - 500 BZA,1,benzaldehyde_106,106.0,105.926159,6.610202e+07
1,(1) 100 BPA - 500 BZA,1,bpa_diacetate_MH_313,313.0,313.274195,2.466208e+09
2,(1) 100 BPA - 500 BZA,1,bpa_diacetate_fragment_270,270.0,270.074717,3.536360e+07
3,(1) 100 BPA - 500 BZA,1,bpa_fragment_213,213.0,212.975652,6.809784e+07
4,(1) 100 BPA - 500 BZA,2,benzaldehyde_106,106.0,105.925757,1.083357e+08
5,(1) 100 BPA - 500 BZA,2,bpa_diacetate_MH_313,313.0,313.274134,2.067617e+09
6,(1) 100 BPA - 500 BZA,2,bpa_diacetate_fragment_270,270.0,270.075994,3.935934e+07
7,(1) 100 BPA - 500 BZA,2,bpa_fragment_213,213.0,213.172068,9.768909e+07
8,(1) 100 BPA - 900 BZA,1,benzaldehyde_106,106.0,NaN,0.000000e+00
9,(1) 100 BPA - 900 BZA,1,bpa_diacetate_MH_313,313.0,313.274552,2.426080e+09


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=peak_check_df)

## Scan-by-scan ratio analysis

In [ ]:
RATIO_NUM = "benzaldehyde_106"
RATIO_DEN = "bpa_fragment_213"

DEN_FRAC_THRESHOLD = 0.01
TRIM_FRAC = 0.05

scan_ratio_rows = []

for (sample, fn), sub in traces_df.groupby(["sample", "function"]):
    sub = sub.sort_values("rt_s").copy()

    num = sub[RATIO_NUM].to_numpy(dtype=float)
    den = sub[RATIO_DEN].to_numpy(dtype=float)
    rt = sub["rt_s"].to_numpy(dtype=float)

    if len(sub) == 0 or np.max(den) <= 0:
        continue

    den_threshold = DEN_FRAC_THRESHOLD * np.max(den)

    keep = den > den_threshold
    num_k = num[keep]
    den_k = den[keep]
    rt_k = rt[keep]

    if len(num_k) == 0:
        continue

    ratio = num_k / den_k

    lo = np.quantile(ratio, TRIM_FRAC)
    hi = np.quantile(ratio, 1 - TRIM_FRAC)

    trim_mask = (ratio >= lo) & (ratio <= hi)
    ratio_trim = ratio[trim_mask]
    rt_trim = rt_k[trim_mask]

    if len(ratio_trim) == 0:
        continue

    mean_ratio = float(np.mean(ratio_trim))
    sd_ratio = float(np.std(ratio_trim, ddof=1)) if len(ratio_trim) > 1 else 0.0
    cv_ratio = float(sd_ratio / mean_ratio) if mean_ratio != 0 else np.nan
    median_ratio = float(np.median(ratio_trim))

    scan_ratio_rows.append({
        "sample": sample,
        "function": fn,
        "n_scans_total": len(sub),
        "n_scans_den_kept": int(np.sum(keep)),
        "n_scans_trimmed_kept": len(ratio_trim),
        "den_threshold": float(den_threshold),
        "trimmed_mean_ratio_106_213": mean_ratio,
        "trimmed_sd_ratio_106_213": sd_ratio,
        "trimmed_cv_ratio_106_213": cv_ratio,
        "trimmed_median_ratio_106_213": median_ratio,
    })

scan_ratio_df = pd.DataFrame(scan_ratio_rows).sort_values(["sample", "function"]).reset_index(drop=True)
scan_ratio_df.head(20)

NameError: name 'traces_df' is not defined

## Compare integrated ratio vs trimmed scan-ratio

In [ ]:
integrated_ratio_df = summary_df.pivot_table(
    index=["sample", "function"],
    columns="target",
    values="integrated_signal_0_60s",
    aggfunc="first"
).reset_index()

integrated_ratio_df["integrated_ratio_106_213"] = (
    integrated_ratio_df["benzaldehyde_106"] / integrated_ratio_df["bpa_fragment_213"]
)

comparison_df = integrated_ratio_df.merge(
    scan_ratio_df,
    on=["sample", "function"],
    how="inner"
)

comparison_df[[
    "sample",
    "function",
    "integrated_ratio_106_213",
    "trimmed_mean_ratio_106_213",
    "trimmed_sd_ratio_106_213",
    "trimmed_cv_ratio_106_213",
    "n_scans_trimmed_kept",
]].head(20)

,sample,function,integrated_ratio_106_213,trimmed_mean_ratio_106_213,trimmed_sd_ratio_106_213,trimmed_cv_ratio_106_213,n_scans_trimmed_kept
0,(1) 100 BPA - 500 BZA,1,0.966434,0.523098,0.521855,0.997624,214
1,(1) 100 BPA - 500 BZA,2,1.214301,1.319909,0.886942,0.671972,225
2,(1) 100 BPA - 900 BZA,1,0.376658,0.300060,0.211668,0.705417,239
3,(1) 100 BPA - 900 BZA,2,0.679078,0.614291,0.285369,0.464549,266
4,(1) 100 BPA - 900 Tol,1,0.377739,0.246014,0.189907,0.771937,272
5,(1) 100 BPA - 900 Tol,2,0.687412,0.542447,0.299802,0.552684,270
6,(2) 100 BPA - 500 BZA,1,1.660699,0.396985,0.517147,1.302685,277
7,(2) 100 BPA - 500 BZA,2,1.718872,0.584455,0.587310,1.004886,270
8,(2) 100 BPA - 900 BZA,1,0.463980,0.310167,0.246656,0.795236,242
9,(2) 100 BPA - 900 BZA,2,0.603277,0.487551,0.228922,0.469534,269


## Method comparison summary by function

In [ ]:
method_summary_df = comparison_df.groupby("function").agg(
    n=("sample", "count"),
    integrated_ratio_mean=("integrated_ratio_106_213", "mean"),
    integrated_ratio_sd=("integrated_ratio_106_213", "std"),
    trimmed_ratio_mean=("trimmed_mean_ratio_106_213", "mean"),
    trimmed_ratio_sd_between_runs=("trimmed_mean_ratio_106_213", "std"),
    mean_within_run_trimmed_sd=("trimmed_sd_ratio_106_213", "mean"),
    mean_within_run_trimmed_cv=("trimmed_cv_ratio_106_213", "mean"),
).reset_index()

method_summary_df

,function,n,integrated_ratio_mean,integrated_ratio_sd,trimmed_ratio_mean,trimmed_ratio_sd_between_runs,mean_within_run_trimmed_sd,mean_within_run_trimmed_cv
0,1,30,0.690959,0.905907,0.480179,0.607802,0.503973,1.011387
1,2,30,0.896238,1.613947,0.798235,1.688248,0.640026,0.840225


## Function 1 per-file comparison table

In [ ]:
f1_scan_ratio_table = comparison_df[comparison_df["function"] == 1][[
    "sample",
    "integrated_ratio_106_213",
    "trimmed_mean_ratio_106_213",
    "trimmed_sd_ratio_106_213",
    "trimmed_cv_ratio_106_213",
    "n_scans_trimmed_kept",
]].sort_values("sample").reset_index(drop=True)

f1_scan_ratio_table

,sample,integrated_ratio_106_213,trimmed_mean_ratio_106_213,trimmed_sd_ratio_106_213,trimmed_cv_ratio_106_213,n_scans_trimmed_kept
0,(1) 100 BPA - 500 BZA,0.966434,0.523098,0.521855,0.997624,214
1,(1) 100 BPA - 900 BZA,0.376658,0.300060,0.211668,0.705417,239
2,(1) 100 BPA - 900 Tol,0.377739,0.246014,0.189907,0.771937,272
3,(2) 100 BPA - 500 BZA,1.660699,0.396985,0.517147,1.302685,277
4,(2) 100 BPA - 900 BZA,0.463980,0.310167,0.246656,0.795236,242
5,(2) 100 BPA - 900 Tol,0.876452,0.469296,0.464073,0.988869,100
6,(3) 100 BPA - 500 BZA,0.306538,0.206416,0.196277,0.950882,271
7,(3) 100 BPA - 900 BZA,0.422106,0.334596,0.217918,0.651288,271
8,(3) 100 BPA - 900 Tol,0.457520,0.345534,0.172435,0.499038,268
9,100 BPA - 100 BZA - 800 Tol,0.063840,0.177835,0.255367,1.435979,19


## List available samples

In [ ]:
available_samples = sorted(traces_df["sample"].unique())
print("Samples:")
for s in available_samples:
    print(" -", s)

Samples:
 - (1) 100 BPA - 500 BZA
 - (1) 100 BPA - 900 BZA
 - (1) 100 BPA - 900 Tol
 - (2) 100 BPA - 500 BZA
 - (2) 100 BPA - 900 BZA
 - (2) 100 BPA - 900 Tol
 - (3) 100 BPA - 500 BZA
 - (3) 100 BPA - 900 BZA
 - (3) 100 BPA - 900 Tol
 - 100 BPA - 100 BZA - 800 Tol
 - 100 BPA - 250 BZA - 650 Tol
 - 100 BPA - 500 BZA - 400 Tol
 - 100 BPA - 750 BZA - 150 Tol
 - 100 BPA - 900 BZA
 - 100 BZA - 900 Tol
 - 500 BPA - 100 BZA - 400 Tol
 - 500 BPA - 250 BZA - 250 Tol
 - 500 BPA - 500 BZA
 - APT
 - APT (1) 100 BPA - 900 BZA
 - APT (1) 100 BPA - 900 Tol
 - BCA Candle 
 - BCW_toluene
 - BCW_toluene_temp
 - Bispehnol Standard Test 
 - Bl (1) 100 BPA - 900 BZA
 - Bl (1) 100 BPA - 900 Tol
 - Bl (2) 100 BPA - 900 Tol
 - Blank Candle
 - White_toluene


## Choose a sample to inspect

In [ ]:
SAMPLE_TO_PLOT = available_samples[0]
print("Plotting:", SAMPLE_TO_PLOT)

## TIC and XIC plots for the chosen sample

In [ ]:
for fn in FUNCTIONS_TO_USE:
    sub = traces_df[(traces_df["sample"] == SAMPLE_TO_PLOT) & (traces_df["function"] == fn)].sort_values("rt_s")
    if len(sub) == 0:
        continue

    plt.figure(figsize=(10, 4))
    plt.plot(sub["rt_s"], sub["tic"])
    plt.xlabel("Time (s)")
    plt.ylabel("Intensity")
    plt.title(f"{SAMPLE_TO_PLOT} — Function {fn} TIC")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    for label in TARGETS:
        plt.plot(sub["rt_s"], sub[label], label=label)
    plt.xlabel("Time (s)")
    plt.ylabel("XIC intensity (summed in window)")
    plt.title(f"{SAMPLE_TO_PLOT} — Function {fn} XICs (±{MZ_TOL} Da)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Background-subtracted spectrum with centroids for the chosen sample

In [ ]:
for fn in FUNCTIONS_TO_USE:
    key = (SAMPLE_TO_PLOT, fn)
    if key not in bgsub_spectra:
        continue

    d = bgsub_spectra[key]
    x = d["mz"]
    y = d["background_subtracted"]
    c_mz = d["centroid_mz"]
    c_int = d["centroid_intensity"]

    plt.figure(figsize=(12, 4))
    plt.plot(x, y, linewidth=1, label="profile-like bg-sub spectrum")
    if len(c_mz):
        plt.scatter(c_mz, c_int, s=18, label="centroids")
    plt.xlim(*MZ_RANGE)
    plt.xlabel("m/z")
    plt.ylabel("Background-subtracted summed intensity")
    plt.title(f"{SAMPLE_TO_PLOT} — Function {fn} bg-sub spectrum with centroids")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    mask = ((x >= 100) & (x <= 110)) | ((x >= 208) & (x <= 215)) | ((x >= 267) & (x <= 273)) | ((x >= 310) & (x <= 315))
    plt.plot(x[mask], y[mask], linewidth=1)
    if len(c_mz):
        c_mask = ((c_mz >= 100) & (c_mz <= 110)) | ((c_mz >= 208) & (c_mz <= 215)) | ((c_mz >= 267) & (c_mz <= 273)) | ((c_mz >= 310) & (c_mz <= 315))
        plt.scatter(c_mz[c_mask], c_int[c_mask], s=20)
    plt.xlabel("m/z")
    plt.ylabel("Background-subtracted summed intensity")
    plt.title(f"{SAMPLE_TO_PLOT} — target-region zoom")
    plt.tight_layout()
    plt.show()

## Export CSV results

In [ ]:
out_dir = DATA_DIR / "colab_outputs"
out_dir.mkdir(exist_ok=True)

traces_path = out_dir / "xic_traces.csv"
summary_path = out_dir / "quant_summary_0_60s.csv"
peakcheck_path = out_dir / "peak_check_bgsub_centroids.csv"
scan_ratio_path = out_dir / "scan_ratio_summary_106_213.csv"
comparison_path = out_dir / "integrated_vs_trimmed_ratio_106_213.csv"
method_summary_path = out_dir / "ratio_method_summary_by_function.csv"

traces_df.to_csv(traces_path, index=False)
summary_df.to_csv(summary_path, index=False)
peak_check_df.to_csv(peakcheck_path, index=False)
scan_ratio_df.to_csv(scan_ratio_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
method_summary_df.to_csv(method_summary_path, index=False)

print("Wrote:")
print(traces_path)
print(summary_path)
print(peakcheck_path)
print(scan_ratio_path)
print(comparison_path)
print(method_summary_path)

Wrote:
/content/drive/MyDrive/Colab Notebooks/Radian/colab_outputs/xic_traces.csv
/content/drive/MyDrive/Colab Notebooks/Radian/colab_outputs/quant_summary_0_60s.csv
/content/drive/MyDrive/Colab Notebooks/Radian/colab_outputs/peak_check_bgsub_centroids.csv
/content/drive/MyDrive/Colab Notebooks/Radian/colab_outputs/scan_ratio_summary_106_213.csv
/content/drive/MyDrive/Colab Notebooks/Radian/colab_outputs/integrated_vs_trimmed_ratio_106_213.csv
/content/drive/MyDrive/Colab Notebooks/Radian/colab_outputs/ratio_method_summary_by_function.csv


## Optional comparison table

In [ ]:
peak_check_df.pivot_table(
    index=["sample", "function"],
    columns="target",
    values="observed_centroid_mz",
    aggfunc="first"
)

target                                benzaldehyde_106  bpa_diacetate_MH_313  \
sample                      function                                           
(1) 100 BPA - 500 BZA       1               105.926159            313.274195   
                            2               105.925757            313.274134   
(1) 100 BPA - 900 BZA       1                      NaN            313.274552   
                            2               105.926857            313.274460   
(1) 100 BPA - 900 Tol       1                      NaN            313.274539   
                            2               105.928703            313.274305   
(2) 100 BPA - 500 BZA       1               105.926336            313.274559   
                            2               105.926302            313.274363   
(2) 100 BPA - 900 BZA       1                      NaN            313.274820   
                            2               105.926662            313.274599   
(2) 100 BPA - 900 Tol       1               106.120788            313.274436   
                            2               106.121599            313.274353   
(3) 100 BPA - 500 BZA       1                      NaN            313.274816   
                            2               106.100042            313.274309   
(3) 100 BPA - 900 BZA       1                      NaN            313.274830   
                            2               105.925775            313.274910   
(3) 100 BPA - 900 Tol       1               106.118775            313.274639   
                            2               105.928958            313.274344   
100 BPA - 100 BZA - 800 Tol 1               105.503350            313.074424   
                            2               105.503451            313.075032   
100 BPA - 250 BZA - 650 Tol 1               105.503741            313.075125   
                            2               105.500861            313.075059   
100 BPA - 500 BZA - 400 Tol 1               105.501577            313.074898   
                            2               105.500697            313.074477   
100 BPA - 750 BZA - 150 Tol 1               105.725502                   NaN   
                            2               105.927849                   NaN   
100 BPA - 900 BZA           1                      NaN            313.069603   
                            2               105.900319                   NaN   
100 BZA - 900 Tol           1                      NaN            313.275256   
                            2                      NaN            313.081055   
500 BPA - 100 BZA - 400 Tol 1               105.505764                   NaN   
500 BPA - 250 BZA - 250 Tol 1               105.503343            313.075316   
                            2               105.500991            313.075410   
500 BPA - 500 BZA           1               105.501877            313.075012   
                            2               105.502190            313.075617   
APT                         1                      NaN            313.274979   
                            2                      NaN            313.274451   
APT (1) 100 BPA - 900 BZA   1               106.061117            313.076251   
                            2               106.122211            313.274726   
APT (1) 100 BPA - 900 Tol   1               105.925486            313.274956   
                            2               105.925886            313.274793   
BCA Candle                  1                      NaN            313.275083   
                            2               106.133644            313.274504   
BCW_toluene                 1               106.122243            313.274631   
                            2               106.122088            313.274361   
BCW_toluene_temp            1               106.075000                   NaN   
                            2               106.075000                   NaN   
Bispehnol Standard Test     1                      NaN            313.074812   


## Notes for future adaptation

Common edits:

### Change which functions are used
```python
FUNCTIONS_TO_USE = [1, 2]
```

### Change target ions
```python
TARGETS = {
    ...
}
```

### Change the XIC extraction window
```python
MZ_TOL = 0.5
```

### Change analysis or background windows
```python
ANALYSIS_END_S = 60.0
BACKGROUND_END_S = 10.0
```

### Change scan-ratio trimming behavior
```python
DEN_FRAC_THRESHOLD = 0.01
TRIM_FRAC = 0.05
```